In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("Presiden Prabowo Menjawab.csv")
df

,Date,Username,Comment,Like
0,2026-03-21T18:53:55Z,@ahmadsyahrulBMX,a jawab b,0
1,2026-03-21T18:52:56Z,@williamlie2207,Jumlah kelas menengah menurun penyebabnya adal...,0
2,2026-03-21T18:52:35Z,@FeriSasono-u1u,dari pertanyaan retno aja jawabannya udah gak ...,0
3,2026-03-21T18:52:20Z,@endraelbumi7577,Jawabannya panjang lebar tp kurang nyambung. N...,0
4,2026-03-21T18:48:57Z,@fajrihidayatullah-sv9cl,Buktinya mana wowo???,0
...,...,...,...,...
9309,2026-03-19T14:22:08Z,@wedustuek8297,Apalagi Ganjar Pranowo presiden pasti jadi beb...,1
9310,2026-03-19T14:03:03Z,@LPKHANAKOREAChanel,Hadir..,0
9311,2026-03-19T14:02:53Z,@ASEPSUPRIATNA-v3r,2,1
9312,2026-03-19T14:02:50Z,@syahrulafani7693,2,1


In [3]:
df = df[['Date', 'Comment']]

In [4]:
df

,Date,Comment
0,2026-03-21T18:53:55Z,a jawab b
1,2026-03-21T18:52:56Z,Jumlah kelas menengah menurun penyebabnya adal...
2,2026-03-21T18:52:35Z,dari pertanyaan retno aja jawabannya udah gak ...
3,2026-03-21T18:52:20Z,Jawabannya panjang lebar tp kurang nyambung. N...
4,2026-03-21T18:48:57Z,Buktinya mana wowo???
...,...,...
9309,2026-03-19T14:22:08Z,Apalagi Ganjar Pranowo presiden pasti jadi beb...
9310,2026-03-19T14:03:03Z,Hadir..
9311,2026-03-19T14:02:53Z,2
9312,2026-03-19T14:02:50Z,2


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9314 entries, 0 to 9313
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     9314 non-null   object
 1   Comment  9314 non-null   object
dtypes: object(2)
memory usage: 145.7+ KB


# Membersihkan data

In [6]:
df = df.copy()

df['Comment'] = (
    df['Comment']
    .astype(str)
    .str.lower()
    .replace('false', pd.NA)
)

# hapus nAn
df = df.dropna(subset=['Comment'])
# Hapus string kosong
# df = df[df['Comment'].str.strip() != '']

In [7]:
df.shape

(9314, 2)

In [8]:
df = df.drop_duplicates(subset=['Comment'])

In [9]:
df.shape

(9088, 2)

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df = df.dropna()

In [12]:
df.shape

(9088, 2)

In [13]:
df.isnull().sum()

Date       0
Comment    0
dtype: int64

In [14]:
df.shape

(9088, 2)

In [15]:
def clean_yt_text(text):
  text = re.sub(r'<.*?>', '', text)  # hapus HTML tag (penting!)
  text = re.sub(r'@[A-Za-z0-9_]+', '', text)  # hapus mention seperti @username
  text = re.sub(r'#\w+', '', text) # hapus hashtag seperti #viral
  text = re.sub(r'https?://\S+', '', text) # hapus URL/link (http/https)

  text = re.sub(r'(.)\1{2,}', r'\1', text)  # ubah huruf berulang (contoh: baguuuus → bagus)
  text = re.sub(r'[^a-zA-Z ]', ' ', text) # hapus karakter selain huruf dan spasi
  text = re.sub(r'\s+', ' ', text).strip() # hapus spasi berlebih & rapikan

  return text

df['Comment'] = df['Comment'].apply(clean_yt_text)

In [16]:
df.shape

(9088, 2)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9088 entries, 0 to 9313
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     9088 non-null   object
 1   Comment  9088 non-null   object
dtypes: object(2)
memory usage: 213.0+ KB


## filter sesuai jumlah huruf dalam kata

In [18]:
# Definisi fungsi filter_tokens_by_length
def filter_tokens_by_length(dataframe, column, min_words, max_words):
    # Tokenisasi kata
    words_count = dataframe[column].astype(str).apply(lambda x: len(x.split()))
    # Membuat filter untuk jumlah kata
    mask = (words_count >= min_words) & (words_count <= max_words)
    # Mengaplikasikan filter ke DataFrame
    filtered_df = dataframe[mask]
    return filtered_df

# Menggunakan filter_tokens_by_length untuk mendapatkan baris dengan jumlah kata antara 3 dan 50
min_words = 3
max_words = 50
df = filter_tokens_by_length(df, 'Comment', min_words, max_words)

In [19]:
df

,Date,Comment
0,2026-03-21T18:53:55Z,a jawab b
2,2026-03-21T18:52:35Z,dari pertanyaan retno aja jawabannya udah gak ...
3,2026-03-21T18:52:20Z,jawabannya panjang lebar tp kurang nyambung ng...
4,2026-03-21T18:48:57Z,buktinya mana wowo
7,2026-03-21T18:45:05Z,apakah anda tahu pnm membantu masyarakat atau ...
...,...,...
9305,2026-03-19T14:06:52Z,sorry bang di sekolah gua bermanfaat
9306,2026-03-19T14:12:27Z,manfaatnya pendidikan gratis lebih ngena
9307,2026-03-19T14:12:45Z,mbg dari kami yang bayar pajak klw gak dari ma...
9308,2026-03-19T14:21:11Z,mbg itu apa hanya dari pajak anda


In [20]:
df.to_csv("hasilCleaning.csv", index=False)